# Stress Testing: Resilience to Sensor Blindness
This notebook simulates a critical hardware failure (loss of HPT Exit Temp telemetry S11) migrating into vector space on flight regime FD004. We demonstrate how the combined **LSTM MC-Dropout uncertainty bounds** and the standalone **Physics Guard** catch catastrophic data faults in real-time.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import json

sys.path.append(os.path.abspath('..'))
from src.data_processor import VectorSequenceProcessor
from src.engine import VectorLSTM

### 1. Load Healthy Engine Sequence (FD004)

In [ ]:
processor = VectorSequenceProcessor()
engine = VectorLSTM()

# Test data for FD004 (Complex conditions and high fault rates)
try:
    df_raw = processor.load_data('../data/test_FD004.txt')
    df_norm = processor.normalize(df_raw, fit=True)
    
    # Extract a pristine 50-cycle sequence from Engine Unit #1
    engine_data = df_norm[df_norm['engine_id'] == 1]
    healthy_sequence = engine_data[processor.sensor_cols].values[-50:]
    print("Healthy Sequence Extracted: Unit 1. Shape:", healthy_sequence.shape)
    
except Exception as e:
    print("Failed to load FD004 Test Data. Proceeding with dummy sequence for code-validation.", e)
    healthy_sequence = np.random.rand(50, 21)

### 2. Establish Baseline Prediction (Healthy Engine Validation)

In [ ]:
print("Evaluating Baseline Healthy State...")
healthy_report = engine.diagnose_health(healthy_sequence)
print(json.dumps(healthy_report, indent=4))

### 3. Simulate Sensor Blindness (S11 Dead Element - Constant 0)

In [ ]:
# Copy sequence to avoid overwriting
corrupted_sequence = np.copy(healthy_sequence)

# Sensor 11 belongs inside index 10.
# We set it explicitly to 0.0 to replicate hardware circuitry loss or thermocouple snap.
corrupted_sequence[:, 10] = 0.0  

print("Evaluating Corrupted Hardware State (S11 Blindness)...")
corrupted_report = engine.diagnose_health(corrupted_sequence)
print(json.dumps(corrupted_report, indent=4))

### 4. System Response Analysis

In [ ]:
# Visualizing the deviation and the catching framework
print("--- VECTOR SYSTEM INTEGRITY LOG ---")
if corrupted_report['physics_status'] == 'Failed':
    print("✅ SUCCESS: Physics Guard securely locked out corrupted prediction.")
    for flag in corrupted_report['warning_flags']:
        print(f"-> {flag}")
else:
    print("❌ FAILURE: Physics Engine failed to secure anomaly constraint.")